In [1]:
from pytential.sympy_pytential import sympy_pytential
from pytential.reduce import min_pytential
import numpy as np
from sympy import log, symbols
import plotly.graph_objects as go
from plotly.subplots import make_subplots

This file demonstrates how to create, combine, and manipulate pytentials.  Equibilriation is used to reduce free variables. 

# Create phases

Create two binary ideal solutions, using the penalization method to affect the lattice constraint via the elastic energy. 

In [2]:
c0a, c1a, c0b, c1b, Va, Vb = symbols('c0a, c1a, c0b, c1b, Va, Vb')

In [3]:
RT = 8.134*300
kappa = 1000
fa_sp = c0a*RT*(2+log(c0a/(c0a+c1a))) + c1a*RT*(0+log(c1a/(c0a+c1a)))+(c0a+c1a)*kappa/2*(log(Va/(c0a+c1a)))**2
fb_sp = c0b*RT*(0+log(c0b/(c0b+c1b))) + c1b*RT*(1+log(c1b/(c0b+c1b)))+(c0b+c1b)*kappa/2*(log(Vb/(c0b+c1b)))**2

In [4]:
# Build the pytentials
fa = sympy_pytential(fa_sp)
fb = sympy_pytential(fb_sp)

print(fa)

x = ['Va', 'c0a', 'c1a']

f(x) = 2440.2*c0a*(log(c0a/(c0a + c1a)) + 2) + 2440.2*c1a*log(c1a/(c0a + c1a)) + (500*c0a + 500*c1a)*log(Va/(c0a + c1a))**2

f'(x)= [2*(500*c0a + 500*c1a)*log(Va/(c0a + c1a))/Va, -2440.2*c1a/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c0a/(c0a + c1a)**2 + 1/(c0a + c1a)) + 500*log(Va/(c0a + c1a))**2 + 2440.2*log(c0a/(c0a + c1a)) + 4880.4 - 2*(500*c0a + 500*c1a)*log(Va/(c0a + c1a))/(c0a + c1a), -2440.2*c0a/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c1a/(c0a + c1a)**2 + 1/(c0a + c1a)) + 500*log(Va/(c0a + c1a))**2 + 2440.2*log(c1a/(c0a + c1a)) - 2*(500*c0a + 500*c1a)*log(Va/(c0a + c1a))/(c0a + c1a)]

f"(x)= [[-2*(500*c0a + 500*c1a)*log(Va/(c0a + c1a))/Va**2 + 2*(500*c0a + 500*c1a)/Va**2, 1000*log(Va/(c0a + c1a))/Va - 2*(500*c0a + 500*c1a)/(Va*(c0a + c1a)), 1000*log(Va/(c0a + c1a))/Va - 2*(500*c0a + 500*c1a)/(Va*(c0a + c1a))], [1000*log(Va/(c0a + c1a))/Va - 2*(500*c0a + 500*c1a)/(Va*(c0a + c1a)), -2440.2*c0a/(c0a + c1a)**2 + 2440.2*c1a/(c0a + c1a)**2 + 2440.2*(c0a + c1a)*(2*c0a/(c0

The elastic strain relaxes the lattice constraint. For reference, we can visualize $f^a$ and $f^b$ assuming the lattice constraint still holds; $c_0^a+c_1^a = V^a = 1$. 

In [5]:
x_values = np.linspace(0.001, .999, 100)
ya = fa(c0a=x_values, c1a=1-x_values, Va = 1)
yb = fb(c0b=x_values, c1b=1-x_values, Vb = 1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_values, y=ya, mode='lines', name='fa'))
fig.add_trace(go.Scatter(x=x_values, y=yb, mode='lines', name='fb'))
fig.update_layout(
    xaxis_title='x',
    yaxis_title='Energy',
    title='fa and fb',
    legend_title='Function'
)
fig.show()

Now we examine the complete space of $f^a$ and $f^b$ and show the previous curves as the valley in the surface: 

In [6]:
# Create meshgrid
X, Y = np.meshgrid(x_values, x_values)

# Calculate Z values for fa and fb
Za = fa(Va=1, c0a=X.ravel(), c1a=Y.ravel()).reshape(X.shape)
Zb = fb(Vb=1, c0b=X.ravel(), c1b=Y.ravel()).reshape(X.shape)

# Line values for the constraint c0a + c1a = 1 (i.e., c1a = 1 - c0a)
fa_line = ya
fb_line = yb

# Create subplots for side-by-side surfaces
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('fa(Va=1)', 'fb(Vb=1)'),
    specs=[[{'type': 'surface'}, {'type': 'surface'}]]
)

# Add surface traces
fig.add_trace(go.Surface(z=Za, x=x_values, y=x_values, name='fa surface', showscale=False), row=1, col=1)
fig.add_trace(go.Surface(z=Zb, x=x_values, y=x_values, name='fb surface', showscale=False), row=1, col=2)

# Add line traces on top of each surface
fig.add_trace(go.Scatter3d(x=x_values, y=1-x_values, z=fa_line,
    mode='lines', line=dict(color='red', width=6), name='fa line'), row=1, col=1)

fig.add_trace(go.Scatter3d(x=x_values, y=1-x_values, z=fb_line,
    mode='lines', line=dict(color='blue', width=6), name='fb line'), row=1, col=2)

# Update layout for both subplots
fig.update_layout(
    title='Side-by-Side Surface Plots of fa and fb with Constraint Line',
    scene1=dict(xaxis_title='c0a', yaxis_title='c1a', zaxis_title='fa'),
    scene2=dict(xaxis_title='c0b', yaxis_title='c1b', zaxis_title='fb'),
)

fig.show()

# Combine functions

We now combine both functions into a composite pytential, and add constraints for the total of each species. 

Note the pytential takes c0 and c1 as arguments even though they only appear in the constraints. 

In [7]:
f = fa+fb
c0, c1 = symbols('c0, c1')
f = f.add_constraints_sym([c0a+c0b-c0, c1a+c1b-c1, Va+Vb-1]) 
print(f)

x = ['Va', 'Vb', 'c0', 'c0a', 'c0b', 'c1', 'c1a', 'c1b']

f(x) = 2440.2*c0a*(log(c0a/(c0a + c1a)) + 2) + 2440.2*c0b*log(c0b/(c0b + c1b)) + 2440.2*c1a*log(c1a/(c0a + c1a)) + 2440.2*c1b*(log(c1b/(c0b + c1b)) + 1) + (500*c0a + 500*c1a)*log(Va/(c0a + c1a))**2 + (500*c0b + 500*c1b)*log(Vb/(c0b + c1b))**2

f'(x)= [2*(500*c0a + 500*c1a)*log(Va/(c0a + c1a))/Va, 2*(500*c0b + 500*c1b)*log(Vb/(c0b + c1b))/Vb, 0, -2440.2*c1a/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c0a/(c0a + c1a)**2 + 1/(c0a + c1a)) + 500*log(Va/(c0a + c1a))**2 + 2440.2*log(c0a/(c0a + c1a)) + 4880.4 - 2*(500*c0a + 500*c1a)*log(Va/(c0a + c1a))/(c0a + c1a), -2440.2*c1b/(c0b + c1b) + 2440.2*(c0b + c1b)*(-c0b/(c0b + c1b)**2 + 1/(c0b + c1b)) + 500*log(Vb/(c0b + c1b))**2 + 2440.2*log(c0b/(c0b + c1b)) - 2*(500*c0b + 500*c1b)*log(Vb/(c0b + c1b))/(c0b + c1b), 0, -2440.2*c0a/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c1a/(c0a + c1a)**2 + 1/(c0a + c1a)) + 500*log(Va/(c0a + c1a))**2 + 2440.2*log(c1a/(c0a + c1a)) - 2*(500*c0a + 500*c1a)*log(Va/(c0a + c1a))

We can now explore the minimizer capabilities. If we minimize over volume, we will find the lowest common tangent. If we keep it fixed at 0 or 1 we will recover the end members. 

In [13]:
f_min_V = min_pytential(f, ['c0', 'c1', 'Va'])
f_min = min_pytential(f, ['c0', 'c1'])

In [14]:
x_values2 = np.linspace(0.001, .999, 20)
ym = f_min(c0=x_values2, c1 = 1-x_values2)
ymV = f_min_V(c0=x_values2, c1 = 1-x_values2, Va= 1)

In [15]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_values, y=ya, mode='lines', name='fa'))
fig.add_trace(go.Scatter(x=x_values, y=yb, mode='lines', name='fb'))
fig.add_trace(go.Scatter(x=x_values2, y=ym, mode='lines', name='f_min'))
fig.add_trace(go.Scatter(x=x_values2, y=ymV, mode='markers', name='f_min_V'))
fig.update_layout(
    xaxis_title='x',
    yaxis_title='Energy',
    title='fa and fb',
    legend_title='Function'
)
fig.show()

We can see what is happening by looking in 3D.

In [16]:
# Create a meshgrid for the two arguments
X, Y = np.meshgrid(x_values2, x_values2)
Z = f_min_V(c0=X.ravel(), c1 = 1-X.ravel(), Va=Y.ravel()).reshape(X.shape)

In [17]:
fig = go.Figure(data=[go.Surface(z=Z, x=x_values2, y=x_values2, colorscale='Viridis')])

# Add a line plot for f_a at Va = 1
fa_values_Va_1 = fa(c0a=x_values, c1a=1-x_values, Va=1)
fig.add_trace(go.Scatter3d(
    x=x_values,
    y=[1] * len(x_values),  # Va = 1
    z=fa_values_Va_1,
    mode='lines',
    name='f_a at Va=1',
    line=dict(color='blue')
))

# Add a line plot for f_b at Va = 0
fb_values_Va_0 = fb(c0b=x_values, c1b=1-x_values, Vb=1)
fig.add_trace(go.Scatter3d(
    x=x_values,
    y=[0] * len(x_values),  # Va = 0
    z=fb_values_Va_0,
    mode='lines',
    name='f_b at Va=0',
    line=dict(color='green')
))

# Add labels and title
fig.update_layout(
    title="Surface Plot of f_min2",
    scene=dict(
        xaxis_title="c0",
        yaxis_title="Va",
        zaxis_title="f_min2",
    ),
)


fig.show()

A convenient way to find the solubility limits (equilibrium state) is to minimize over ca and cb:

In [34]:
f_min.min_fcn(np.array([[0.5, 0.5]]))


(array([-639.02570437, -639.02570437]),
 [{'c0': 0.5,
   'Va': 0.45085199543413934,
   'Vb': 0.5491480045658605,
   'c0a': 0.049999822107249785,
   'c0b': 0.45000017789275026,
   'c1': 0.7318114704863753,
   'c1a': 0.5053648652248498,
   'c1b': 0.22644660526152555},
  {'c0': 0.5,
   'Va': 0.45085199543413934,
   'Vb': 0.5491480045658605,
   'c0a': 0.049999822107249785,
   'c0b': 0.45000017789275026,
   'c1': 0.7318114704863753,
   'c1a': 0.5053648652248498,
   'c1b': 0.22644660526152555}])